# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields by @id

print("Record Sets available in this dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"- Record Set name: {record_set.name} | @id: {record_set.id}")
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            print(f"    - Field: {field.name} | @id: {field.id} | Data Type: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify all record set @ids
record_set_ids = [record_set.id for record_set in dataset.metadata.record_sets]
print("Loaded RecordSet @ids:", record_set_ids)

# Prepare DataFrames for each RecordSet
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from {record_set_id}")
    except Exception as e:
        print(f"Error loading '{record_set_id}': {e}")

# Preview the first available DataFrame
if dataframes:
    first_record_set_id = record_set_ids[0]
    print(f"Columns in DataFrame for {first_record_set_id}:\n", dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set and a numeric field for analysis, referencing by @id
# Replace these with the specific @id's discovered above as appropriate for this dataset

# Example: select the first record set and a numeric field from its fields
# You can update 'numeric_field_id' and 'group_field_id' below to match the IDs from your dataset overview

selected_record_set_id = record_set_ids[0] if record_set_ids else None

if selected_record_set_id is not None:
    field_ids = []
    group_candidate_ids = []
    numeric_candidate_ids = []
    # Find fields for the selected record set
    for rs in dataset.metadata.record_sets:
        if rs.id == selected_record_set_id:
            for field in rs.fields:
                field_ids.append(field.id)
                # Guess numeric fields by data type
                if field.data_type in ['Float','Integer','Number']:
                    numeric_candidate_ids.append(field.id)
                else:
                    group_candidate_ids.append(field.id)
    print("Numeric candidate field @ids:", numeric_candidate_ids)
    print("Group candidate field @ids:", group_candidate_ids)

    # Pick the first available numeric field and group field
    numeric_field_id = numeric_candidate_ids[0] if numeric_candidate_ids else field_ids[0] if field_ids else None
    group_field_id = group_candidate_ids[0] if group_candidate_ids else None

    print(f"Using RecordSet: {selected_record_set_id}, Numeric Field: {numeric_field_id}, Group Field: {group_field_id}")
    df = dataframes[selected_record_set_id]

    # Filter, normalize, group operations
    threshold = 10
    # It's possible the column labels are the field names, not IDs. Let's check both.
    if numeric_field_id in df.columns:
        field_col = numeric_field_id
    elif hasattr(df, 'columns') and len(df.columns)>0:
        field_col = df.columns[0]
        print(f"Numeric field id '{numeric_field_id}' not found, use first column: {field_col}")
    else:
        field_col = None

    if field_col is not None and pd.api.types.is_numeric_dtype(df[field_col]):
        filtered_df = df[df[field_col] > threshold].copy()
        print(f"Filtered records with {field_col} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{field_col}_normalized"] = (filtered_df[field_col] - filtered_df[field_col].mean()) / filtered_df[field_col].std()
        print(f"Normalized {field_col} for filtered records:")
        print(filtered_df[[field_col, f"{field_col}_normalized"]].head())

        # Try grouping if group field exists
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[field_col].mean().reset_index()
            print(f"Grouped data by {group_field_id}")
            print(grouped_df.head())
    else:
        print("No numeric field available for EDA in the selected record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# Example: histogram and boxplot for the numeric field, if available
if 'filtered_df' in locals() and field_col is not None and not filtered_df.empty:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[field_col].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {field_col} (filtered)')

    plt.subplot(1,2,2)
    sns.boxplot(x=filtered_df[field_col].dropna())
    plt.title(f'Boxplot of {field_col} (filtered)')
    plt.tight_layout()
    plt.show()
else:
    print('No suitable numeric data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load, examine, and process records from the mass spectrometry dataset using `mlcroissant`. Using the Croissant record sets and field `@id`s, we've inspected available fields, loaded data into pandas DataFrames, performed filtering and normalization on numeric fields, grouped data, and generated basic visualizations. This approach supports FAIR, transparent access and analysis for biomedical data science workflows.